# 16.2 A scheduling model

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In [Section 2.4](../02/2_4_generalizations_to_blending_economics_and_scheduling.ipynb) we observed that the general form of a blending model was applicable
to certain scheduling problems. Here we describe a related scheduling model for which
the columnwise approach is particularly attractive.

Suppose that a factory's production for the next week is divided into fixed time periods,
or shifts. You want to assign employees to shifts, so that the required number of
people are working on each shift. You cannot fill each shift independently of the others,
however, because only certain weekly schedules are allowed; for example, a person cannot
work five shifts in a row. Your problem is thus more properly viewed as one of
assigning employees to schedules, so that each shift is covered and the overall assignment
is the most economical.

We can conveniently represent the schedules for this problem by an indexed collection
of subsets of shifts:

```ampl
set SHIFTS;               # shifts
param Nsched;             # number of schedules;
set SCHEDS = 1..Nsched;   # set of schedules
set SHIFT_LIST {SCHEDS} within SHIFTS;
```

For each schedule `j` in the set SCHEDS, the shifts that a person works on schedule `j` are
contained in the set SHIFT_LIST[j]. We also specify a pay rate per person on each
schedule, and the number of people required on each shift:

```ampl
param rate {SCHEDS} >= 0;
param required {SHIFTS} >= 0;
```

We let the variable Work[j] represent the number of people assigned to work each
schedule j, and minimize the sum of rate[j] * Work[j] over all schedules:

```ampl
var Work {SCHEDS} >= 0;
minimize Total_Cost: sum {j in SCHEDS} rate[j] * Work[j];
```

Finally, our constraints say that the total of employees assigned to each shift `i` must be at
least the number required:

```ampl
subject to Shift_Needs {i in SHIFTS}:
       sum {j in SCHEDS: i in SHIFT_LIST[j]} Work[j]
       >= required[i];
```

On the left we take the sum of Work[j] over all schedules `j` such that `i` is in
SHIFT_LIST[j]. This sum represents the total employees who are assigned to schedules
that contain shift i, and hence equals the total employees covering shift i.

The awkward description of the constraint in this formulation motivates us to try a
columnwise formulation. As in our previous examples, we declare the objective and constraints
first, but with the variables left out:

```ampl
minimize Total_Cost;
subject to Shift_Needs {i in SHIFTS}: to_come >= required[i];
```

The coefficients of Work[j] appear instead in its `var` declaration. In the objective, it
has a coefficient of rate[j]. In the constraints, the membership of SHIFT_LIST[j]
tells us exactly what we need to know: Work[j] has a coefficient of 1 in constraint
Shift_Needs[i] for each `i` in SHIFT_LIST[j], and a coefficient of 0 in the other
constraints. This leads us to the following concise declaration:

```ampl
var Work {j in SCHEDS} >= 0,
       obj Total_Cost rate[j],
       coeff {i in SHIFT_LIST[j]} Shift_Needs[i] 1;
```

The full model is shown in [Figure 16-4](./16_2_a_scheduling_model.ipynb#fig-16-4).

As a specific instance of this model, imagine that you have three shifts a day on Monday
through Friday, and two shifts on Saturday. Each day you need 100, 78 and 52
employees on the first, second and third shifts, respectively. To keep things simple, suppose
that the cost per person is the same regardless of schedule, so that you may just minimize
the total number of employees by setting rate[j] to 1 for all j.

As for the schedules, a reasonable scheduling rule might be that each employee works
five shifts in a week, but never more than one shift in any 24-hour period. Part of the
data file is shown in [Figure 16-5](./16_2_a_scheduling_model.ipynb#fig-16-5); we don't show the whole file, because there are 126
schedules that satisfy the rule! The resulting 126-variable linear program is not hard to
solve, however:

```ampl
ampl: model sched.mod; data sched.dat; solve;
MINOS 5.5: optimal solution found.

19 iterations, objective 265.6
ampl: option display_eps .000001;
ampl: option omit_zero_rows 1;
ampl: option display_1col 0, display_width 60;
ampl: display Work;
Work [*] :=
       10 28.8   30 14.4   71 35.6    106 23.2  123 35.6
       18 7.6    35 6.8    73 28      109 14.4
       24 6.8    66 35.6   87 14.4    113 14.4
;
```

As you can see, this optimal solution makes use of 13 of the schedules, some in fractional
amounts. (There exist many other optimal solutions to this problem, so the results you
get may differ.) If you round each fraction in this solution up to the next highest value,
you get a pretty good feasible solution using 271 employees. To determine whether this
is the best whole-number solution, however, it is necessary to use integer programming
techniques, which are the subject of [Chapter 20](../20/20.md).

The convenience of the columnwise formulation in this case follows directly from
how we have chosen to represent the data. We imagine that the modeler will be thinking
in terms of schedules, and will want to try adding, dropping or modifying different schedules
to see what solutions can be obtained. The subsets SHIFT_LIST[j] provide a
convenient and concise way of maintaining the schedules in the data. Since the data are
then organized by schedules, and there is also a variable for each schedule, it proves to be
simpler — and for larger examples, more efficient — to specify the coefficients by variable.

```ampl
set SHIFTS;             # shifts
param Nsched;           # number of schedules;
set SCHEDS = 1..Nsched; # set of schedules
set SHIFT_LIST {SCHEDS} within SHIFTS;
param rate {SCHEDS} >= 0;
param required {SHIFTS} >= 0;
minimize Total_Cost;
subject to Shift_Needs {i in SHIFTS}: to_come >= required[i];
var Work {j in SCHEDS} >= 0,
       obj Total_Cost rate[j],
       coeff {i in SHIFT_LIST[j]} Shift_Needs[i] 1;
```

**[Figure 16-4](./16_2_a_scheduling_model.ipynb#fig-16-4):** Columnwise scheduling model (sched.mod).

```ampl
set SHIFTS := Mon1 Tue1 Wed1 Thu1 Fri1 Sat1
	    Mon2 Tue2 Wed2 Thu2 Fri2 Sat2
	    Mon3 Tue3 Wed3 Thu3 Fri3 ;
param Nsched := 126 ;
set        SHIFT_LIST[1] := Mon1 Tue1 Wed1 Thu1 Fri1 ;
set        SHIFT_LIST[2] := Mon1 Tue1 Wed1 Thu1 Fri2 ;
set        SHIFT_LIST[3] := Mon1 Tue1 Wed1 Thu1 Fri3 ;
set        SHIFT_LIST[4] := Mon1 Tue1 Wed1 Thu1 Sat1 ;
set        SHIFT_LIST[5] := Mon1 Tue1 Wed1 Thu1 Sat2 ;
(117 lines omitted)
set        SHIFT_LIST[123] := Tue1  Wed1  Thu1  Fri2  Sat2 ;
set        SHIFT_LIST[124] := Tue1  Wed1  Thu2  Fri2  Sat2 ;
set        SHIFT_LIST[125] := Tue1  Wed2  Thu2  Fri2  Sat2 ;
set        SHIFT_LIST[126] := Tue2  Wed2  Thu2  Fri2  Sat2 ;
param rate default 1 ;
param required :=  Mon1  100  Mon2  78 Mon3  52
                   Tue1  100  Tue2  78 Tue3  52
                   Wed1  100  Wed2  78 Wed3  52
                   Thu1  100  Thu2  78 Thu3  52
                   Fri1  100  Fri2  78 Fri3  52
                   Sat1  100  Sat2  78 ;
```

**[Figure 16-5](./16_2_a_scheduling_model.ipynb#fig-16-5):** Partial data for scheduling model (sched.dat).


Models of this kind are used for a variety of scheduling problems. As a convenience,
the keyword `cover` may be used (in the manner of from and to for networks) to specify
a coefficient of 1:

```ampl
var Work {j in SCHEDS} >= 0,
       obj Total_Cost rate[j],
       cover {i in SHIFT_LIST[j]} Shift_Needs[i];
```

Some of the best known and largest examples are in airline crew scheduling, where the
variables may represent the assignment of crews rather than individuals, the shifts
become flights, and the requirement is one crew for each flight. We then have what is
known as a set covering problem, in which the objective is to most economically `cover`
the set of all flights with subsets representing crew schedules.